#**PDF Chatbox**

In [2]:
#Installing the Libraries

!pip uninstall -y \
langchain \
langchain-core \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
langsmith

!pip install \
langchain==0.3.27 \
langchain-core==0.3.72 \
langchain-community==0.3.27 \
langchain-google-genai==2.1.8 \
langchain-text-splitters==0.3.9 \
pypdf \
faiss-cpu

Found existing installation: langchain 1.3.13
Uninstalling langchain-1.3.13:
  Successfully uninstalled langchain-1.3.13
Found existing installation: langchain-core 1.4.9
Uninstalling langchain-core-1.4.9:
  Successfully uninstalled langchain-core-1.4.9
Found existing installation: langsmith 0.10.2
Uninstalling langsmith-0.10.2:
  Successfully uninstalled langsmith-0.10.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 675.2/675.2 kB 33.8 MB/s eta 0:

In [1]:
#Importing the Libraries

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [2]:
#Storing My API Key
api_key=userdata.get("My_Key")

In [3]:
#Loading my PDF and Extracting Text

loader=PyPDFLoader("Data_Science Notes.pdf")
document=loader.load()
print(document)

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': 'Data_Science Notes.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='1 ⃣  PYTHON  –  Answers  \nWhat  is  the  difference  between  list,  tuple,  set,  dictionary?  \n●  List :  Ordered,  mutable,  allows  duplicates  \n ●  Tuple :  Ordered,  immutable,  faster  than  list  \n ●  Set :  Unordered,  mutable,  no  duplicates  \n ●  Dictionary :  Key–value  pairs,  mutable,  fast  lookup   \nMutable  vs  Immutable  objects?  \n●  Mutable :  Can  be  changed  after  creation  (list,  dict,  set)  \n ●  Immutable :  Cannot  be  changed  (int,  float,  string,  tuple)   \nNone vs  NaN  \n●  None :  Python  object  meaning  “no  value”  \n ●  NaN :  Numerical  missing  value  (used  in  NumPy/Pandas)               \n     Shallow  copy  vs  Deep  copy  \n●  Shallow  copy :  Copies  reference,  changes  affect  original  \n ● 

In [4]:
#Splitting the Text

chunks=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200).split_documents(document)

In [5]:
#Creating Embeddings

embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key)
vector_stores=FAISS.from_documents(chunks,embeddings)

In [6]:
#Retriever

retriever=vector_stores.as_retriever()

In [7]:
#LLM

llm=ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=api_key,
    temperature=0
)

In [8]:
#Prompt Generation

prompt=ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the user's question only using the provided context.

If the answer is not available in the context,
say "I couldn't find the answer in the uploaded PDF."
Context:
{context}

Question:
{input}
""")

In [9]:
#Document Chain

document_chain=create_stuff_documents_chain(
    llm,prompt
)

In [10]:
#Retreiver Chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)
response=retrieval_chain.invoke({
    "input" : "What is this pdf about?"
})
print(response)

{'input': 'What is this pdf about?', 'context': [Document(id='0069d8f4-f0bb-4cc2-841e-434b3c93c229', metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': 'Data_Science Notes.pdf', 'total_pages': 9, 'page': 6, 'page_label': '7'}, page_content='6 ⃣  NLP  –  Answers  \nNLP  \n●  Processing  human  language   \nTokenization  \n●  Splitting  text  into  words   \nStemming  vs  Lemmatization  \n●  Stemming:  crude  chopping  \n ●  Lemmatization:  dictionary-based   \nBag  of  Words  \n●  Word  frequency  representation   \nTF-IDF  \n●  Importance  of  word  in  document   \nWord  embedding  \n●  Dense  vector  representation   \nCosine  similarity  \n●  Measures  angle  between  vectors   \n  7 ⃣  UNSUPERVISED  –  Answers  \nClustering  \n●  Grouping  similar  data   \nKMeans  steps  \n●  Initialize  →  assign  →  update  →  repeat   \nChoosing  K  \n●  Elbow  method'), Document(id='d15146d0-efad-4534-ac63

In [11]:
#Response keys

print(response.keys())

dict_keys(['input', 'context', 'answer'])


In [12]:
#Generating Answer

print(response['answer'])

Based on the provided context, the document contains concise definitions and key concepts related to the following topics:

* **NLP (Natural Language Processing):** Tokenization, stemming vs. lemmatization, Bag of Words, TF-IDF, word embeddings, and cosine similarity.
* **Unsupervised Learning:** Clustering, KMeans steps, and choosing K (elbow method).
* **EDA (Exploratory Data Analysis) and Statistics:** Missing values, outliers, feature engineering, data leakage, normalization vs. standardization, correlation vs. covariance, normal distribution, Central Limit Theorem, p-values, hypothesis testing, Type I & II errors, confidence intervals, and Z-test vs. T-test.
* **Machine Learning:** What ML is and types of ML (Supervised, Unsupervised, Reinforcement).
* **Python Concepts:** Lambda functions, decorators, list comprehension, and `append()` vs. `extend()`.
